In [0]:


%sql
CREATE SCHEMA IF NOT EXISTS medical_pipeline.silver;

In [0]:
from pyspark.sql.functions import regexp_replace , col , unix_timestamp
proc_df = spark.table("medical_pipeline.bronze.procedures")
proc_df= proc_df.select('start',
 'stop',
 'patient',
 'encounter_id',
 'code',
 'description',
 'base_cost',
 'reasoncode',
 'reasondescription')

proc_df = proc_df.withColumn(
    "base_cost",
    regexp_replace("base_cost", ",", "").cast("double")
)
proc_df = proc_df.fillna({
    "reasondescription": "unknown",
    "description": "unknown"
})
proc_df = proc_df.dropDuplicates(["patient", "encounter_id", "start", "code"])
proc_df = proc_df.filter(
    col("patient").isNotNull() &
    col("encounter_id").isNotNull() &
    col("start").isNotNull()
)

proc_df = proc_df.withColumn(
    "duration_minutes",
    (unix_timestamp("stop") - unix_timestamp("start")) / 60
)



In [0]:


proc_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("medical_pipeline.silver.procedures")

In [0]:
%sql
select * from medical_pipeline.silver.procedures;

In [0]:
%sql
select * from medical_pipeline.silver.encounters_silver;